# Prompt Engineering Lab: From Vague Requests to Reliable Results

**Duration:** 45 minutes  
**Format:** Work in pairs. Choose **one main task** first; start a second only after completing two prompt iterations.

**Nothing here is graded.** There are no points, no scores, and no grades in this lab. The goal is simply practice, so aim for:
- diagnosing a failure,
- improving the prompt deliberately,
- and explaining why Prompt V2 is better than Prompt V1.

A first output that fails is useful — it gives you something to fix.


## Lab schedule

| Time | Activity |
|---|---|
| 0–5 min | Choose a challenge and form pairs |
| 5–10 min | Run a weak baseline prompt; identify failures |
| 10–25 min | Create and test Prompt V1 |
| 25–35 min | Review the output and build Prompt V2 |
| 35–42 min | Final polish or begin another challenge |
| 42–45 min | Lightning demos and reflection |


## Prompt Engineering Canvas

A strong prompt often includes:

1. **Role** — relevant expertise or perspective.
2. **Goal** — a concrete deliverable.
3. **Context** — source text, user profile, or background.
4. **Constraints** — requirements, exclusions, safety, audience, length.
5. **Examples (in-context learning)** — demonstrate the desired transformation or format.
6. **Process** — plan → draft → check → revise.
7. **Output contract** — exact structure, JSON schema, code format, or headings.
8. **Verification** — rubric, tests, edge cases, or a checklist.

> For reasoning, request a **brief plan and verification summary**. Do not require the model to reveal private hidden reasoning.


## Experiment log

This log is only for your own tracking — it is never collected or graded. For every run, note:

| Version | What changed? | Biggest success | Biggest failure | What to try next |
|---|---|---|---|---|
| Baseline |  |  |  |  |
| V1 |  |  |  |  |
| V2 |  |  |  |  |

Self-check questions (not a score): Is the goal clear? Is the needed context there? Are constraints and safety explicit? Are there useful examples? Is the output format specified and verified?


## Setup — run both cells below before any challenge

**On Colab:** click **File → Save a copy in Drive** first. A notebook opened straight from GitHub is read-only, and anything you type is lost when the runtime disconnects.

You will call the model directly from this notebook. Register at [console.groq.com](https://console.groq.com), create your own API key, and paste it into `API_KEY` in the second cell.

### Interaction rules

**Challenges 1–4 are single-shot: one prompt in, one response out, no follow-up messages.**
Every run starts from a blank slate, so any improvement you see comes from the prompt itself rather than from the chat history. If an output is wrong, fix the prompt and run again — do not argue with the model.

**Challenge 5 is the exception.** Building a game is a steering exercise, so that challenge uses a real multi-turn conversation and a different helper function.


In [ ]:
# Setup 1 of 2 — environment bootstrap. Run this first.
# On Colab this fetches the assets/ folder and installs the client.
# On your own machine it does nothing except confirm where you are.

import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = "https://github.com/shilianghe007/llm-workshop-highschool.git"
    if not os.path.exists("/content/lab"):
        subprocess.run(["git", "clone", "-q", REPO, "/content/lab"], check=True)
    os.chdir("/content/lab/prompt_engineering_lab")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai"], check=True)

print("working dir:", os.getcwd())
print("assets:", sorted(os.listdir("assets")))


In [ ]:
# Setup 2 of 2 — API client and helper functions.

import re
import sys
from html import escape
from pathlib import Path

from IPython.display import HTML, display
from openai import OpenAI

IN_COLAB = "google.colab" in sys.modules

# Paste your own key between the quotes.
API_KEY = "PASTE_YOUR_KEY_HERE"

client = OpenAI(api_key=API_KEY, base_url="https://api.groq.com/openai/v1")

MODEL = "openai/gpt-oss-20b"


def ask(prompt, model=MODEL):
    """Single-shot: send ONE prompt with no conversation history, return the reply text."""
    return client.responses.create(input=prompt, model=model).output_text


def save_html(reply, filename="index.html"):
    """Pull the last code block out of a reply and write it to disk."""
    blocks = re.findall(r"```(?:html)?\s*\n(.*?)```", reply, re.DOTALL)
    html = blocks[-1] if blocks else reply
    Path(filename).write_text(html)
    print(f"Wrote {filename} ({len(html)} characters).")
    return html


def show_html(html, height=600):
    """Preview a page inside this notebook. Good enough to look at it and play a game."""
    display(HTML(
        f'<iframe srcdoc="{escape(html)}" width="100%" height="{height}" '
        'style="border:1px solid #ccc"></iframe>'
    ))


def download(filename):
    """Send the file to your computer so you can test it in a real browser."""
    if IN_COLAB:
        from google.colab import files
        files.download(filename)
    else:
        print(f"Not on Colab — just open {Path(filename).resolve()}")


print(ask("Reply with exactly: setup works"))


# Challenge 1 — Style Remix With Public-Domain Songs

Open `assets/public_domain_song_style_pack.md`.

**Goal:** Write a new song about debugging, summer camp, or lost Wi-Fi by abstracting the traits of one reference style.

### Weak baseline
> Write a catchy song about debugging in the style of a folk song.

### Requirements
Your prompt must make the model:
- infer a compact style guide from the references before writing anything,
- treat the reference pack as in-context examples,
- avoid reusing any complete reference line,
- follow the verse/chorus structure, approximate line length, rhyme behavior, and repetition that you specify,
- produce Draft 1, then a checklist critique, then a revised Draft 2,
- and keep the final song suitable for a school activity.

### Stretch goal
Ask for two variants that preserve the same style rules but differ in mood.


In [ ]:
baseline_prompt = "Write a catchy song about debugging in the style of a folk song."

prompt_v1 = r"""
PASTE YOUR PROMPT V1 HERE
"""

prompt_v2 = r"""
PASTE YOUR PROMPT V2 HERE
"""

# Each call is independent — the model remembers nothing between versions.
song = {}

song["baseline"] = ask(baseline_prompt)
print(song["baseline"])

# Once Prompt V1 is written, run it and compare:
# song["v1"] = ask(prompt_v1)
# print(song["v1"])

# Then Prompt V2:
# song["v2"] = ask(prompt_v2)
# print(song["v2"])


# Challenge 2 — Generate a Personal Homepage

Open:
- `assets/homepage_brief.md`
- `assets/starter_homepage.html`

**Goal:** Prompt an LLM to produce a polished, accessible, standalone `index.html`.

### Weak baseline
> Make Sam a cool personal website.

### Requirements
Your prompt must make the model:
- keep all HTML, CSS, and JavaScript in one file,
- turn every item in the client brief into a visible page element,
- meet the responsive and accessibility requirements that you spell out,
- work against a code-quality checklist that you supply,
- verify that navigation, the email link, keyboard focus, and the chosen interaction all work,
- and output a short implementation plan followed by exactly one complete HTML code block.


In [ ]:
baseline_prompt = "Make Sam a cool personal website."

homepage_prompt_v1 = r"""
PASTE YOUR HOMEPAGE PROMPT V1 HERE
"""

reply = ask(baseline_prompt)          # swap in homepage_prompt_v1 when it is ready
print(reply)

html = save_html(reply)                # writes index.html
show_html(html)                        # quick preview inside the notebook

# The preview is not enough for the accessibility checks — the page needs a real browser.
# Run download("index.html"), open the downloaded file, then:
# 1. Resize the window down to phone width.
# 2. Tab through the page with the keyboard only.
# 3. Click the email link and try the interactive feature.
# Whatever fails here is the reason for Prompt V2.


# Challenge 3 — Personalized Fitness and Meal Plan

Open `assets/fitness_profiles.json` and choose Profile A, B, or C.

**Goal:** Produce a realistic one-week plan that respects every constraint.

### Weak baseline
> Give this student a fitness and diet plan.

### Requirements
Your prompt must make the model:
- separate facts from assumptions explicitly,
- build a weekly schedule that fits the available time and equipment,
- cover warm-up, main activity, cooldown, intensity guidance, and recovery,
- give meal *patterns and examples* rather than extreme calorie targets,
- respect allergies, dietary constraints, preferences, school schedule, and age,
- supply substitutions and a "what to do if a session is missed" rule,
- end with a compliance table mapping each profile constraint to where it was handled,
- and keep the advice general, recommending a qualified adult or professional for medical concerns.


In [ ]:
import json

# The file holds three profiles; pick one by id.
profiles = {p["id"]: p for p in json.loads(Path("assets/fitness_profiles.json").read_text())}
profile = profiles["A"]                # choose "A", "B", or "C"
print(json.dumps(profile, indent=2))

baseline_prompt = "Give this student a fitness and diet plan."

fitness_prompt_v1 = f"""
PASTE YOUR INSTRUCTIONS HERE

PROFILE:
```json
{json.dumps(profile, indent=2)}
```
"""

plan = ask(baseline_prompt)            # swap in fitness_prompt_v1 when it is ready
print(plan)


# Challenge 4 — Messy Text to Validated JSON

Open:
- `assets/messy_club_signups.txt`
- `assets/extraction_schema.json`

**Goal:** Extract all student records into JSON without inventing missing details.

### Weak baseline
> Turn these notes into JSON.

### Requirements
Your prompt must make the model:
- follow the supplied schema exactly,
- normalize phone numbers and shirt sizes,
- distinguish missing, uncertain, and conflicting information,
- resolve a conflict only when the text provides enough evidence, and otherwise record it,
- represent uncertainty the way at least one few-shot example you provide demonstrates,
- validate required keys, types, allowed values, and record count before the final output,
- and emit valid JSON only in the final block.

### Adversarial test
After Prompt V2 works, alter one detail in the messy text or add a contradictory note. Does the prompt still prevent hallucination?


In [ ]:
import json

messy = Path("assets/messy_club_signups.txt").read_text()
schema_text = Path("assets/extraction_schema.json").read_text()

baseline_prompt = f"Turn these notes into JSON.\n\n{messy}"

extraction_prompt_v1 = f"""
PASTE INSTRUCTIONS HERE

SCHEMA:
```json
{schema_text}
```

SOURCE NOTES:
```text
{messy}
```
"""

reply = ask(baseline_prompt)           # swap in extraction_prompt_v1 when it is ready
print(reply[:2000])


def check_json(reply):
    """Automated check: is the final block parseable, and how many records came back?"""
    blocks = re.findall(r"```(?:json)?\s*\n(.*?)```", reply, re.DOTALL)
    try:
        data = json.loads(blocks[-1] if blocks else reply)
    except json.JSONDecodeError as e:
        print(f"INVALID JSON — {e}")
        return None
    students = data.get("students", []) if isinstance(data, dict) else data
    print(f"Valid JSON — {len(students)} student records.")
    return data


data = check_json(reply)


# Challenge 5 — Build a Browser Game

Open `assets/game_ideas.md` and select one idea.

**Goal:** Prompt an LLM to produce a playable game that runs from one standalone `index.html`.

> **This is the multi-turn challenge.** Challenges 1–4 were single-shot. Here you steer the model across several turns, and every turn carries the full conversation history. Use `chat()` instead of `ask()`.

### Weak baseline
> Make me a fun browser game.

### Requirements
Your prompt must define:
- game loop and exact rules,
- controls,
- start / pause / restart behavior,
- scoring and difficulty,
- win or loss conditions,
- accessibility and mobile behavior,
- a self-test checklist,
- and the requirement that everything runs in one standalone `index.html`.

### Suggested turn-by-turn workflow
1. **Turn 1** — ask for a compact technical design. No code yet.
2. **Turn 2** — ask the model to list every rule that is still ambiguous.
3. **Turn 3** — decide those rules yourself and send your decisions back.
4. **Turn 4** — ask for the complete `index.html`.
5. **Turn 5+** — play-test, then report each bug with exact symptoms: what you did, what happened, what you expected.

Note which turn produced the biggest jump in quality — that is the interesting result, not the finished game.


In [ ]:
# Challenge 5 uses a CONVERSATION: every call resends the whole history.
game_messages = []


def chat(message, model=MODEL):
    """Multi-turn: append your message, resend the full history, keep the reply."""
    game_messages.append({"role": "user", "content": message})
    reply = client.responses.create(input=game_messages, model=model).output_text
    game_messages.append({"role": "assistant", "content": reply})
    return reply


def reset_chat():
    """Start the conversation over from scratch."""
    game_messages.clear()
    print("Conversation cleared.")


# Turn 1 — ask for a technical design, not code.
turn_1 = r"""
PASTE YOUR TURN-1 PROMPT HERE
"""

print(chat(turn_1))


In [ ]:
# Re-run this cell for every following turn: edit the message, run it, read the reply.
next_message = r"""
YOUR NEXT MESSAGE HERE
"""

print(chat(next_message))
print(f"\n--- conversation is now {len(game_messages)} messages long ---")

# Once the model has produced the game, play it without leaving the notebook:
#   html = save_html(game_messages[-1]["content"], "game.html")
#   show_html(html, height=700)
#
# For keyboard and mobile testing, get it into a real browser:
#   download("game.html")
#
# Then report each bug in the next turn.
# reset_chat() starts a clean conversation if you want to try a different approach.


# Final reflection

Answer briefly:

1. Which prompt component produced the largest improvement?
2. Where did the model still make assumptions?
3. Did examples improve **content**, **format**, or both?
4. What automated check would make your prompt more reliable?
5. What would you change for a different LLM?
